In [1]:
import pandas as pd
import numpy as np

# Product Statistics

In [2]:
product = pd.read_excel(r"..\Raw data\Product-Statistics-UPI-Product-statistics-upi\2026-27-Monthly.xlsx")
product.head()

,Month,No. of Banks live on UPI,Volume (In Mn.),Value (In Cr.)
0,October-2026,758,"25,466.70","31,52,929.46"
1,September-2026,756,"24,068.38","29,37,396.67"
2,August-2026,752,"24,508.96","29,82,355.95"
3,July-2026,741,"23,658.35","29,87,880.49"
4,June-2026,731,22716.07,"28,92,138.67"


In [3]:
import glob, os

folder = r"..\Raw data\Product-Statistics-UPI-Product-statistics-upi"
files = sorted(glob.glob(os.path.join(folder, "*.xlsx")))

parts = []
for f in files:
    x = pd.read_excel(f, dtype=str)
    x["source_file"] = os.path.basename(f)
    parts.append(x)

combined = pd.concat(parts, ignore_index=True)
combined.to_csv("product_combined_raw.csv", index=False)

print(len(files), "files |", combined.shape)   # expect 8 files, 91 rows
combined.head()

8 files | (91, 5)


,Month,No. of Banks live on UPI,Volume (In Mn.),Value (In Cr.),source_file
0,March-2020,148,1246.84,"2,06,462.31",2019-20-Monthly.xlsx
1,February-2020,146,1325.69,"2,22,516.95",2019-20-Monthly.xlsx
2,January-2020,144,1305.02,"2,16,242.97",2019-20-Monthly.xlsx
3,December-2019,143,1308.4,"2,02,520.76",2019-20-Monthly.xlsx
4,November-2019,143,1218.77,"1,89,229.09",2019-20-Monthly.xlsx


In [4]:
combined.sample(20)

,Month,No. of Banks live on UPI,Volume (In Mn.),Value (In Cr.),source_file
60,March-2025,661\t,"18,301.51\t","24,77,221.61",2024-25-Monthly.xlsx
54,September-2023,492,10555.69,"15,79,133.18",2023-24-Monthly.xlsx
38,January-2023,385,8036.89,"12,98,726.62",2022-23-Monthly.xlsx
43,August-2022,346,6579.63,"10,72,792.68",2022-23-Monthly.xlsx
29,October-2021,261,4218.65,"7,71,444.98",2021-22-Monthly.xlsx
64,November-2024,637,15482.02,"21,55,187.40",2024-25-Monthly.xlsx
14,January-2021,207,2302.73,"4,31,181.89",2020-21-Monthly.xlsx
73,February-2026,694,"20,394.18","26,84,229.29",2025-26-Monthly.xlsx
82,May-2025,673\t,"18,677.46\t","25,14,297.01",2025-26-Monthly.xlsx
61,February-2025,653,16106.19,"21,96,481.69",2024-25-Monthly.xlsx


In [5]:
combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Month                     91 non-null     str  
 1   No. of Banks live on UPI  91 non-null     str  
 2   Volume (In Mn.)           91 non-null     str  
 3   Value (In Cr.)            91 non-null     str  
 4   source_file               91 non-null     str  
dtypes: str(5)
memory usage: 8.4 KB


In [6]:
ws = combined.drop(columns="source_file").apply(lambda c: c.str.contains(r"^\s|\s$", regex=True))
combined[ws.any(axis=1)]

,Month,No. of Banks live on UPI,Volume (In Mn.),Value (In Cr.),source_file
60,March-2025,661\t,"18,301.51\t","24,77,221.61",2024-25-Monthly.xlsx
81,June-2025,675\t,"18,395.01\t","24,03,930.69",2025-26-Monthly.xlsx
82,May-2025,673\t,"18,677.46\t","25,14,297.01",2025-26-Monthly.xlsx
83,April-2025,668\t,"17,893.42\t","23,94,925.87",2025-26-Monthly.xlsx


In [7]:
combined = combined.apply(lambda c: c.str.strip())

In [8]:
clean = combined.copy()
clean = clean.apply(lambda c: c.str.strip())          # removes the stray \t

num_cols = ["No. of Banks live on UPI", "Volume (In Mn.)", "Value (In Cr.)"]
for c in num_cols:
    clean[c] = pd.to_numeric(clean[c].str.replace(",", "", regex=False), errors="coerce")

print(clean[num_cols].isna().sum())                    # expect 0 everywhere
print(clean[num_cols].describe().loc[["min", "max"]])
clean.dtypes

No. of Banks live on UPI    0
Volume (In Mn.)             0
Value (In Cr.)              0
dtype: int64
     No. of Banks live on UPI  Volume (In Mn.)  Value (In Cr.)
min                     141.0           733.54       142034.39
max                     758.0         25466.70      3152929.46


Month                           str
No. of Banks live on UPI      int64
Volume (In Mn.)             float64
Value (In Cr.)              float64
source_file                     str
dtype: object

In [9]:
clean

,Month,No. of Banks live on UPI,Volume (In Mn.),Value (In Cr.),source_file
0,March-2020,148,1246.84,206462.31,2019-20-Monthly.xlsx
1,February-2020,146,1325.69,222516.95,2019-20-Monthly.xlsx
2,January-2020,144,1305.02,216242.97,2019-20-Monthly.xlsx
3,December-2019,143,1308.40,202520.76,2019-20-Monthly.xlsx
4,November-2019,143,1218.77,189229.09,2019-20-Monthly.xlsx
...,...,...,...,...,...
86,August-2026,752,24508.96,2982355.95,2026-27-Monthly.xlsx
87,July-2026,741,23658.35,2987880.49,2026-27-Monthly.xlsx
88,June-2026,731,22716.07,2892138.67,2026-27-Monthly.xlsx
89,May-2026,720,23201.93,2990424.21,2026-27-Monthly.xlsx


In [10]:
clean["Month"] = pd.to_datetime(clean["Month"], format="%B-%Y")
clean["year"] = clean["Month"].dt.year
clean["month_num"] = clean["Month"].dt.month
clean["fy"] = clean["Month"].apply(lambda d: f"{d.year}-{str(d.year + 1)[2:]}" if d.month >= 4 else f"{d.year - 1}-{str(d.year)[2:]}")
clean = clean.sort_values("Month").reset_index(drop=True)
clean.to_excel("product_clean.xlsx", index=False)

In [11]:
out = clean.drop(columns="source_file")
out["Month"] = out["Month"].dt.date          # date only, no 00:00:00
out.to_excel("product_clean.xlsx", index=False)

In [12]:
clean.describe()

,Month,No. of Banks live on UPI,Volume (In Mn.),Value (In Cr.),year,month_num
count,91,91.000000,91.000000,9.100000e+01,91.000000,91.000000
mean,2022-12-31 10:01:19.120879,420.252747,9770.855495,1.394451e+06,2022.538462,6.538462
min,2019-04-01 00:00:00,141.000000,733.540000,1.420344e+05,2019.000000,1.000000
25%,2021-02-15 00:00:00,214.500000,2421.150000,4.609103e+05,2021.000000,4.000000
50%,2023-01-01 00:00:00,385.000000,7829.490000,1.282055e+06,2023.000000,7.000000
75%,2024-11-16 00:00:00,639.000000,16345.580000,2.260591e+06,2024.000000,9.000000
max,2026-10-01 00:00:00,758.000000,25466.700000,3.152929e+06,2026.000000,12.000000
std,NaN,214.016126,7700.504235,9.590118e+05,2.217645,3.383974


In [13]:
print(clean["Month"].is_monotonic_increasing, clean["Month"].duplicated().sum())
print(len(pd.date_range(clean["Month"].min(), clean["Month"].max(), freq="MS")))   # expect 91

True 0
91


# Economic Statistics

### Application Stats

### Application Stats: data notes (source: NPCI UPI Ecosystem Statistics)

**How NPCI defines the numbers**
- App volume uses the *payer app* logic: a transaction is credited to the PSP on the payer's side.
- All B2C and B2B transactions are credited to the respective PSP.
- From Jan 2021, on-us transactions (not settled through the UPI central system) appear in a separate *On-us* column.
- Apps with volume under 20,000 are grouped into an **"Other Apps"** row, so the app list is not exhaustive.
- BHIM volume includes *99# volume.
- Airtel Payments Bank on-us transactions are included under B2B.
- TPAP market-share statistics exclude cross-border transactions and new/recent features.
- `#` marks a **3rd Party App (TPAP)**. `WhatsApp*` is a footnote about onboarding limits being removed on 31 Dec 2024.

**What I found in the data**
- Coverage: Jun 2020 to Sep 2026 (76 months). Oct 2026 is not published yet.
- Jun to Nov 2020 rows use older field names (`upi_application_name`, `upi_volume_mn`, `upi_value_cr`) and have no B2C/B2B/on-us split.
- The `#` tag only appears from Feb 2024, so it can't be used as a per-month TPAP flag.
- App names are inconsistent (case, spacing, stray characters like `Â` and `*`).

**Cleaning decisions** *(fill in as decided)*
| Decision | Reason |
|---|---|
| Strip `#` and `*` from names, keep `has_hash` as the TPAP flag | NPCI note |
| Case/spacing variants merged via a lowercase, symbol-free key | Same app, different spelling |
| Paytm (OCL): _pending_ | Possibly a separate entity from Paytm |
| Paytm Payment(s) Bank spellings: merged | No month has more than one |
| Fam Pay / Fam App by Trio: merged | One value per month across spellings |
| Shriram vs Shriram One: _pending_ | Not yet checked |

**Known issues to resolve**
- Missing `vol_mn` (22 rows) and `val_cr` (207 rows), including Paytm from Sep 2025 to Mar 2026.
- Apps total is far below the headline in Sep 2025.
- 5 duplicate (month, app) rows.

In [14]:
apps = pd.read_csv(r"..\Raw data\upi_apps_monthly.csv", parse_dates=["month"])
print(apps.shape)
print("="*20, "NULL VALUES", "="*20)
print(apps.isna().sum())
print("="*20, "DATA TYPES", "="*20)
apps.info()
print("="*20, "DESCRIPTIVE STATS", "="*20)
apps.describe()


(5410, 14)
==================== NULL VALUES ====================
month             0
app_raw           0
vol_mn           22
val_cr          207
ci_vol_mn       394
ci_val_cr       554
b2c_vol_mn     3857
b2c_val_cr     3802
b2b_vol_mn     3759
b2b_val_cr     3540
onus_vol_mn    4562
onus_val_cr    4540
has_hash          0
app               0
dtype: int64
==================== DATA TYPES ====================
<class 'pandas.DataFrame'>
RangeIndex: 5410 entries, 0 to 5409
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   month        5410 non-null   datetime64[us]
 1   app_raw      5410 non-null   str           
 2   vol_mn       5388 non-null   float64       
 3   val_cr       5203 non-null   float64       
 4   ci_vol_mn    5016 non-null   float64       
 5   ci_val_cr    4856 non-null   float64       
 6   b2c_vol_mn   1553 non-null   float64       
 7   b2c_val_cr   1608 non-null   float64       
 8 

,month,vol_mn,val_cr,ci_vol_mn,ci_val_cr,b2c_vol_mn,b2c_val_cr,b2b_vol_mn,b2b_val_cr,onus_vol_mn,onus_val_cr
count,5410,5388.000000,5.203000e+03,5016.000000,4.856000e+03,1553.000000,1608.000000,1651.000000,1870.000000,848.000000,8.700000e+02
mean,2023-11-07 18:29:24.731977,133.067137,1.799855e+04,138.487373,1.876868e+04,5.050693,225.111623,0.215083,162.057716,47.579375,6.451462e+03
min,2020-06-01 00:00:00,0.010000,2.000000e-02,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00
25%,2022-05-01 00:00:00,0.090000,2.156000e+01,0.100000,2.416250e+01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00
50%,2024-01-01 00:00:00,0.710000,1.147400e+02,0.730000,1.196000e+02,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00
75%,2025-07-01 00:00:00,6.095000,7.452250e+02,6.285000,7.491011e+02,0.010000,0.232500,0.010000,0.557500,0.045000,2.270000e+00
max,2026-09-01 00:00:00,11185.100622,1.467169e+06,11185.100622,1.467169e+06,128.830000,11114.520000,45.530000,12841.440000,8682.080000,1.256541e+06
std,NaN,807.825641,1.109936e+05,835.341698,1.147229e+05,17.735911,1107.123306,1.442273,1070.107103,536.573092,7.389985e+04


In [15]:
apps.sample(20)

,month,app_raw,vol_mn,val_cr,ci_vol_mn,ci_val_cr,b2c_vol_mn,b2c_val_cr,b2b_vol_mn,b2b_val_cr,onus_vol_mn,onus_val_cr,has_hash,app
4957,2026-05-01,Yes Bank Apps,49.630571,7634.859406,3.614340,2614.501628,45.763342,5006.162599,0.252889,14.195178,NaN,NaN,False,Yes Bank Apps
1686,2022-10-01,Citi Bank Apps,0.090000,66.830000,0.090000,66.830000,NaN,NaN,NaN,NaN,NaN,NaN,False,Citi Bank Apps
2003,2023-03-01,IndusInd Bank Apps,0.930000,950.350000,0.930000,950.350000,NaN,NaN,NaN,NaN,NaN,NaN,False,IndusInd Bank Apps
1425,2022-06-01,Bank of Maharashtra Apps,0.060000,15.770000,0.060000,15.770000,NaN,NaN,NaN,NaN,NaN,NaN,False,Bank of Maharashtra Apps
1434,2022-06-01,Allahabad Bank Apps,0.020000,6.790000,0.020000,6.790000,NaN,NaN,NaN,NaN,NaN,NaN,False,Allahabad Bank Apps
3808,2025-04-01,FamApp by Trio #,99.910000,1241.800000,99.910000,1241.800000,NaN,NaN,NaN,NaN,NaN,NaN,True,FamApp by Trio
289,2020-10-01,Janata Sahakari Bank,0.010000,5.970000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,Janata Sahakari Bank
1984,2023-03-01,BHIM,23.680000,7498.240000,23.680000,7498.240000,NaN,NaN,NaN,NaN,NaN,NaN,False,BHIM
1798,2022-12-01,Deutsche Bank Apps,1.280000,104.490000,1.280000,104.470000,NaN,NaN,NaN,0.010000,NaN,NaN,False,Deutsche Bank Apps
5197,2026-07-01,SnapMint #,0.220000,20.060000,0.220000,20.060000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,True,SnapMint


In [20]:
apps["app"].unique()

<ArrowStringArray>
[               'Google Pay',                   'PhonePe',
        'Paytm Payment Bank',                 'AmazonPay',
                      'BHIM',           'ICICI Bank Apps',
             'Yes Bank Apps',  'State Bank of India Apps',
 'Airtel Payments Bank Apps',            'Axis Bank Apps',
 ...
                     'jUMMP',                  'Stashfin',
                     'Novio',                   'Unimoni',
                   'LivQuik',              'Hero Fincorp',
                      'CheQ',                   'REVOLUT',
                    'ENKASH',                  'Mufinpay']
Length: 301, dtype: str

In [21]:
import re, itertools, difflib
import pandas as pd

def norm(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9 ]", "", s)       # drops * . ( ) Â etc.
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+apps?$", "", s)         # trailing App / Apps
    s = re.sub(r"\s+bank$", "", s)          # trailing Bank
    return s.replace(" ", "")

names = pd.Series(sorted(apps["app"].unique()))
key = names.map(norm)

# 1) names that collapse to the same key
groups = names.groupby(key).apply(list)
variants = groups[groups.str.len() > 1]
print(f"{len(variants)} entities with variants | "
      f"{variants.str.len().sum()} names -> {len(variants)}")

# "!!" = two variants appear in the same month, so they may be distinct entities
months = apps.groupby("app")["month"].agg(set)
for k, v in variants.items():
    clash = any(months[a] & months[b] for a, b in itertools.combinations(v, 2))
    print(("!! " if clash else "   ") + str(v))

# 2) fuzzy near-matches among the remaining keys (typos like Fedral / Deustche)
keys = sorted(set(key))
fz = [(round(difflib.SequenceMatcher(None, a, b).ratio(), 2), a, b)
      for a, b in itertools.combinations(keys, 2)]
fz = [p for p in fz if p[0] >= 0.88]
print(len(fz), "fuzzy pairs")
for r, a, b in sorted(fz, reverse=True):
    print(r, groups[a], "<->", groups[b])

72 entities with variants | 177 names -> 72
   ['Allahabad Bank', 'Allahabad Bank App', 'Allahabad Bank Apps']
   ['Amazon Pay', 'AmazonPay']
   ['Andhra Bank', 'Andhra Bank App', 'Andhra Bank Apps']
   ['AU Small Finance Bank App', 'AU Small Finance Bank Apps', 'Au Small Finance Bank Apps']
   ['Bank of Baroda', 'Bank of Baroda Apps']
   ['Bank of India App', 'Bank of India Apps']
   ['Bank of Maharashtra', 'Bank of Maharashtra App', 'Bank of Maharashtra Apps']
   ['Canara Bank', 'Canara Bank App', 'Canara Bank Apps']
   ['CBDC', 'CBDC Apps']
   ['Central Bank of India', 'Central Bank of India App', 'Central Bank of India Apps']
   ['CheQ', 'Cheq']
   ['Citi Bank App', 'Citi Bank Apps']
   ['City Union Bank App', 'City Union Bank Apps']
   ['CRED', 'Cred']
   ['DBS Digibank', 'DBS Digibank App', 'DBS Digibank Apps']
   ['Dena Bank', 'Dena Bank App', 'Dena Bank Apps']
   ['Deutsche Bank', 'Deutsche Bank App', 'Deutsche Bank Apps']
   ['DHANI', 'Dhani']
   ['Dhanlaxmi Bank App', 'Dhanla

In [22]:
# ---- build mapping from the detection output ----
skip_keys = {"bajajfinservppi", "fifederal"}   # fuzzy-similar but different entities
prefer = {"Vijaya Bank"}                       # force the correct spelling as canonical

cnt = apps["app"].value_counts()

# union fuzzy-matched keys into clusters (skipping the ones above)
parent = {k: k for k in groups.index}
def find(k):
    while parent[k] != k:
        k = parent[k]
    return k
for r, a, b in fz:
    if {a, b} & skip_keys:
        continue
    parent[find(a)] = find(b)

# one cluster = every raw name that should become one app
clusters = {}
for k, ns in groups.items():
    clusters.setdefault(find(k), []).extend(ns)

# canonical name = preferred spelling if present, else most frequent spelling
mapping = {}
for ns in clusters.values():
    canon = next((n for n in ns if n in prefer), max(ns, key=lambda n: cnt[n]))
    mapping.update({n: canon for n in ns if n != canon})

# merges the 0.88 fuzzy cutoff can't see (verify these)
extras = {
    "SBI Bank Apps": "State Bank of India Apps",
    "DBS Bank Apps": "DBS Digibank Apps",
    "Punjab And Sind Bank Apps": "Punjab Sind Bank Apps",
    "FamPay by Trio": "FamApp by Trio",
    "Fampay by Trio": "FamApp by Trio",
    "Fam Pay by Trio": "FamApp by Trio",
    "WhatsApp Pay": "WhatsApp",
}
def resolve(n):
    n = extras.get(n, n)
    return mapping.get(n, n)

apps["app_clean"] = apps["app"].map(resolve)
print(apps["app"].nunique(), "->", apps["app_clean"].nunique())

d = apps[apps.duplicated(["month", "app_clean"], keep=False)].sort_values(["app_clean", "month"])
print(len(d), "rows in conflict")
print(d[["month", "app_raw", "app_clean", "vol_mn", "val_cr"]].to_string())

301 -> 183
18 rows in conflict
          month            app_raw          app_clean  vol_mn   val_cr
1923 2023-02-01      Bajaj Finserv      Bajaj Finserv    5.44   882.75
1972 2023-02-01      Bajaj Finserv      Bajaj Finserv    0.01     3.43
3118 2024-07-01  Federal Bank Apps  Federal Bank Apps    5.95   848.66
3121 2024-07-01  Federal Bank Apps  Federal Bank Apps    2.68   460.69
2560 2023-11-01           Mobikwik           Mobikwik    4.52  1467.04
2599 2023-11-01           Mobikwik           Mobikwik    0.06     8.71
2629 2023-12-01           Mobikwik           Mobikwik    4.95  1580.30
2657 2023-12-01           Mobikwik           Mobikwik    0.28    38.53
4175 2025-08-01             Others             Others    2.41   273.01
4177 2025-08-01             Others             Others    2.31   244.14
2426 2023-09-01              slice              slice    1.06   310.90
2461 2023-09-01              SLICE              slice    0.03     3.80
2566 2023-11-01              slice            

In [24]:
apps["app_clean"].unique()

<ArrowStringArray>
[               'Google Pay',                   'PhonePe',
   'Paytm Payments Bank App',                'Amazon Pay',
                      'BHIM',           'ICICI Bank Apps',
             'Yes Bank Apps',  'State Bank of India Apps',
 'Airtel Payments Bank Apps',            'Axis Bank Apps',
 ...
                     'Jummp',                  'SnapMint',
                 'Rio Money',                  'Stashfin',
                     'Novio',                   'Unimoni',
              'Hero Fincorp',                   'REVOLUT',
                    'ENKASH',                  'Mufinpay']
Length: 183, dtype: str